# Google Speech-to-Text for Nepali & English

This notebook builds the **speech-to-text (STT)** component using **Google Cloud Speech-to-Text (Chirp)** instead of a locally-run model like Whisper.

**Why a Google Cloud model instead of a local model on a MacBook Air M2 (8GB RAM):**
- Chirp is Google's universal speech model — it runs entirely on Google's servers. Your laptop only
  records/uploads audio and receives text back, so RAM/CPU usage on your Mac stays tiny (no multi-GB
  model ever gets loaded into memory).
- 8GB of RAM is enough to run `whisper-tiny`/`base` locally, but `small`/`medium` (needed for good Nepali
  accuracy) get slow and memory-heavy on an 8GB machine. A cloud API sidesteps that trade-off entirely —
  it will feel equally fast regardless of your hardware.
- Chirp officially supports **Nepali (`ne-NP`)** and **English (`en-US`)**, including automatic
  language identification between a short list of languages you specify.

**Pipeline covered here:**
1. Install dependencies
2. Set up a Google Cloud project + credentials
3. Transcribe an audio file (Nepali, English, or auto-detect between the two)
4. Record audio directly from the microphone (optional, for live testing)
5. Wrap everything into a reusable `GoogleSpeechToText` class


## 1. Prerequisites (one-time Google Cloud setup)

1. Create/select a project in the [Google Cloud Console](https://console.cloud.google.com/).
2. Enable the **Cloud Speech-to-Text API** for that project (APIs & Services → Enable APIs → search
   "Speech-to-Text API").
3. Create a **service account** (IAM & Admin → Service Accounts), grant it the
   `roles/speech.client` role, and download a JSON key file for it.
4. Note your **project ID** — you'll need it below.

You do **not** need a GPU or any local model download for any of this — everything below just needs a
network connection.


In [ ]:
%pip install -q google-cloud-speech sounddevice scipy numpy

## 2. Authenticate

Point the client library at your service account key. Replace the path below with wherever you saved
the JSON key file.


In [ ]:
import os

# Path to the service account JSON key you downloaded in the prerequisites step
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/path/to/your-service-account-key.json"

PROJECT_ID = "your-gcp-project-id"  # <-- replace with your project ID
LOCATION = "global"  # "global" works for most use cases; some Chirp features need a region like "us-central1"


## 3. Transcribe an audio file

We use the Speech-to-Text **v2 API** with the `chirp` model, which supports Nepali and English (among
~100 languages). Audio should be a standard file (`.wav`, `.flac`, `.mp3`, etc.) — the API auto-detects
the encoding and sample rate.


In [ ]:
from google.cloud import speech_v2
from google.cloud.speech_v2.types import cloud_speech

client = speech_v2.SpeechClient()


def transcribe_file(audio_path: str, language_codes=("en-US",), model="chirp"):
    """Transcribe a local audio file with Google Cloud Speech-to-Text (Chirp).

    language_codes can list more than one BCP-47 code (e.g. ("ne-NP", "en-US"))
    to let the model auto-identify which of those languages was spoken.
    """
    with open(audio_path, "rb") as f:
        content = f.read()

    config = cloud_speech.RecognitionConfig(
        auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
        language_codes=list(language_codes),
        model=model,
    )

    request = cloud_speech.RecognizeRequest(
        recognizer=f"projects/{PROJECT_ID}/locations/{LOCATION}/recognizers/_",
        config=config,
        content=content,
    )

    response = client.recognize(request=request)

    text = " ".join(
        result.alternatives[0].transcript
        for result in response.results
        if result.alternatives
    )
    detected_language = (
        response.results[0].language_code if response.results else None
    )
    return {"text": text.strip(), "language": detected_language}


### Nepali example


In [ ]:
audio_path = "sample_nepali.wav"  # <-- change to your audio file
result_ne = transcribe_file(audio_path, language_codes=["ne-NP"])
print(result_ne["text"])

### English example


In [ ]:
audio_path = "sample_english.wav"  # <-- change to your audio file
result_en = transcribe_file(audio_path, language_codes=["en-US"])
print(result_en["text"])

### Auto-detecting between Nepali and English

Since your assistant serves both languages and won't always know which one is spoken, pass both codes.
Chirp will identify which language was actually used and return it in `language_code`.


In [ ]:
audio_path = "sample.wav"  # <-- change to your audio file
result_auto = transcribe_file(audio_path, language_codes=["ne-NP", "en-US"])
print("Detected language:", result_auto["language"])
print("Transcript:", result_auto["text"])

## 4. (Optional) Record audio live from the microphone

Useful for testing end-to-end without pre-recorded files. Only works when running the notebook locally
with a microphone (won't work on a headless/remote server).


In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write

def record_audio(filename="mic_input.wav", duration=5, samplerate=16000):
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype="int16")
    sd.wait()
    write(filename, samplerate, audio)
    print(f"Saved to {filename}")
    return filename

# recorded_file = record_audio(duration=5)
# result = transcribe_file(recorded_file, language_codes=["ne-NP", "en-US"])
# print(result["text"])

## 5. Reusable `GoogleSpeechToText` class

Wrap the client into a small class so the rest of your assistant (translation, intent detection, TTS
reply, etc.) can just call `.transcribe(path)` without knowing about the Speech-to-Text API internals.


In [ ]:
class GoogleSpeechToText:
    def __init__(self, project_id: str, location: str = "global", model: str = "chirp",
                 language_codes=("ne-NP", "en-US")):
        self.client = speech_v2.SpeechClient()
        self.project_id = project_id
        self.location = location
        self.model = model
        self.language_codes = list(language_codes)

    def transcribe(self, audio_path: str, language_codes=None):
        with open(audio_path, "rb") as f:
            content = f.read()

        config = cloud_speech.RecognitionConfig(
            auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
            language_codes=language_codes or self.language_codes,
            model=self.model,
        )
        request = cloud_speech.RecognizeRequest(
            recognizer=f"projects/{self.project_id}/locations/{self.location}/recognizers/_",
            config=config,
            content=content,
        )
        response = self.client.recognize(request=request)

        text = " ".join(
            r.alternatives[0].transcript for r in response.results if r.alternatives
        )
        detected_language = response.results[0].language_code if response.results else None
        return {"text": text.strip(), "language": detected_language}


# Example usage:
# stt = GoogleSpeechToText(project_id=PROJECT_ID, language_codes=["ne-NP", "en-US"])
# output = stt.transcribe("sample.wav")
# print(output["language"], "->", output["text"])

## Notes: fit for your MacBook Air M2 (8GB RAM)

- No model weights are ever downloaded or loaded on your machine — the heaviest local operation is
  reading a short audio file into memory, so this runs identically fast whether you're on an 8GB Air or
  a 128GB workstation.
- Trade-off vs. the local Whisper notebook in this repo: this approach needs internet access, a Google
  Cloud project, and billing enabled (Speech-to-Text has a free monthly quota; check current pricing on
  the [Speech-to-Text pricing page](https://cloud.google.com/speech-to-text/pricing) before heavy use).
  Whisper trades that for slower/heavier local inference but works fully offline.
- **Simpler alternative:** if setting up a GCP service account is more than you need right now, the
  **Gemini API** (`google-genai` SDK) also accepts audio input directly and can transcribe Nepali/English
  with just an API key — no service account or project setup — at the cost of being a general-purpose
  LLM rather than a dedicated, benchmarked speech model like Chirp.

## Next steps for the assistant pipeline
1. Feed `GoogleSpeechToText().transcribe(audio)["text"]` into your NLP/intent-detection or LLM component.
2. Use the detected `language` to pick a matching text-to-speech voice for the reply.
3. If deploying as a web app, replace the mic-recording cell with a browser audio recorder (e.g. a JS
   `MediaRecorder` widget) that uploads the audio file to this backend.
